定理 2.6 —— 平面上任意 n 条直线（不要求居一般位置）划分出的区域可以只用两种颜色着色，
使相邻区域颜色不同。

input: `lines` 为 (a, b, c) 三元组，表示直线 ax+by+c=0；查询点 `point` (x, y)，不在任何直线上

output: 该点所在区域的颜色（0 或 1）。构造方法直接取自证明：颜色 = 该点位于
多少条直线正侧的计数的奇偶性。跨过一条直线只会翻转这个和里的一项，所以相邻区域
颜色必然不同——这正是证明中“翻转一侧所有区域颜色”那一步在做的事。

In [ ]:
import random


def side(line, point):
    a, b, c = line
    x, y = point
    return 1 if a * x + b * y + c > 0 else 0


def region_color(point, lines):
    return sum(side(l, point) for l in lines) % 2


def verify_two_coloring(lines, trials=200, eps=1e-6, seed=0):
    '''
    取一条直线上的随机点，向两侧各推 eps 距离，两个推出的点必须颜色不同——
    这正是定理所声称的相邻关系。若推出的点离另一条直线太近就跳过，
    因为此时“只有一条直线将它们分开”的前提不再成立。
    '''
    rng = random.Random(seed)
    checked = 0
    for _ in range(trials):
        i = rng.randrange(len(lines))
        a, b, c = lines[i]
        if b != 0:
            x = rng.uniform(-5, 5)
            y = -(a * x + c) / b
        else:
            y = rng.uniform(-5, 5)
            x = -(b * y + c) / a
        norm = (a ** 2 + b ** 2) ** 0.5
        nx, ny = a / norm, b / norm
        p1 = (x + eps * nx, y + eps * ny)
        p2 = (x - eps * nx, y - eps * ny)
        if any(abs(l[0] * x + l[1] * y + l[2]) < 1e-3 for j, l in enumerate(lines) if j != i):
            continue
        assert region_color(p1, lines) != region_color(p2, lines)
        checked += 1
    return checked

In [ ]:
lines = [
    (1, 0, 0),    # x = 0
    (0, 1, 0),    # y = 0
    (1, 1, -3),   # x + y = 3
    (1, -1, 1),   # x - y = -1
]

print("color(2, 2) =", region_color((2, 2), lines))
print("color(-1, -1) =", region_color((-1, -1), lines))

checked = verify_two_coloring(lines)
print(f"共校验 {checked} 次跨线情形，颜色均按定理要求发生翻转")